# 📖 Tracing Basic Agents

---

## 🎯 Learning Objectives

1. Add tracing to agents
2. View and understand traces
3. Debug with traces

---

## ⏱️ Time Estimate
**~25 minutes**

In [ ]:
!pip install -q langsmith langchain langchain-openai
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "agent-evals-tutorial"

if "LANGCHAIN_API_KEY" not in os.environ:
    os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith Key: ")
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI Key: ")

## 💻 Full Traced Agent

In [ ]:
from langchain_core.tools import tool, StructuredTool
from langchain_openai import ChatOpenAI
from langchain.agents import create_openai_agent, AgentExecutor
from langchain_core.messages import SystemMessage
from langsmith import traceable

# Define traced tools
@tool
def get_stock_price(symbol: str) -> str:
    """Get stock price."""
    prices = {"AAPL": "$178.50", "GOOGL": "$142.30", "MSFT": "$378.20"}
    return prices.get(symbol.upper(), "$0.00")

@tool
def calculator(expression: str) -> str:
    """Calculate math expression."""
    try:
        return str(eval(expression))
    except:
        return "Error"

tools = [get_stock_price, calculator]

# Create agent with system message
llm = ChatOpenAI(model="gpt-4o-mini")
agent = create_openai_agent(llm, tools, SystemMessage(content="""You are a helpful assistant. Use tools when needed."""))
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Traced Agent Created!")
print("Run this agent and check LangSmith dashboard!")

In [ ]:
# Run the traced agent
result = agent_executor.invoke("What's the price of AAPL?")
print(f"\n✅ Result: {result['output']}")
print("\n📊 Check LangSmith dashboard for full trace!")

## 📊 What to Look for in Traces

In [ ]:
print("""
┌─────────────────────────────────────────────────────┐
│         TRACE COMPONENTS TO CHECK                  │
├─────────────────────────────────────────────────────┤
│  1. INPUT: What did the user ask?                │
│  2. TOOL DECISION: Did agent call right tool?       │
│  3. TOOL INPUT: Were params correct?            │
│  4. TOOL OUTPUT: Did tool return correctly?       │
│  5. FINAL OUTPUT: Did agent respond?           │
│  6. TOKENS: How many tokens used?               │
└─────────────────────────────────────────────────────┘
```

Common trace issues:
• ❌ Wrong tool called = Check tool descriptions
• ❌ Wrong params = Improve tool schema
• ❌ Tool failed = Check error handling
• ❌ Hallucinated = Check grounding
```

## ✅ Summary

You can now:
1. Add tracing to agents
2. View traces in LangSmith
3. Debug agent issues

**This understanding is CRITICAL for agent evals!**

## 🔗 Next
**[PART_5_Agent_Evals/01_what_are_agent_evals.ipynb](PART_5_Agent_Evals/01_what_are_agent_evals.ipynb)** - Learn what evals are!